<a href="https://colab.research.google.com/github/dudinha-web/fundamentos-de-ia/blob/main/Aula_12_Geopandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introdução ao GeoPandas
### Aula de Ciência de Dados — Dados Geoespaciais com Python

Link Documentação: https://geopandas.org/en/stable/gallery/overlays.html

In [ ]:
!pip install geopandas

In [ ]:
!pip install geopandas mapclassify --upgrade -q

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point, LineString, Polygon

In [ ]:
!pip install GeoPandas

In [ ]:
!pip install geodatasets

---
## Carregando GeoJSON — Estados Brasileiros


In [ ]:
# GeoJSON público dos estados brasileiros
url_estados = 'https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson'

estados = gpd.read_file(url_estados)

print('Shape:', estados.shape)
print('Colunas:', estados.columns.tolist())
print('\nPrimeiras linhas:')
estados[['name', 'sigla', 'geometry']].head()

In [ ]:
# Identificar a região de cada estado
regioes = {
    'Norte':     ['AM','PA','AC','RO','RR','AP','TO'],
    'Nordeste':  ['MA','PI','CE','RN','PB','PE','AL','SE','BA'],
    'Centro-Oeste': ['MT','MS','GO','DF'],
    'Sudeste':   ['SP','RJ','MG','ES'],
    'Sul':       ['PR','SC','RS']
}

sigla_para_regiao = {uf: reg for reg, ufs in regioes.items() for uf in ufs}
estados['regiao'] = estados['sigla'].map(sigla_para_regiao)

print('Estados por região:')
estados['regiao'].value_counts()

In [ ]:
# Mapa colorido por região
cores_regiao = {
    'Norte':        '#2196F3',
    'Nordeste':     '#FF9800',
    'Centro-Oeste': '#9C27B0',
    'Sudeste':      '#F44336',
    'Sul':          '#4CAF50'
}

estados['cor'] = estados['regiao'].map(cores_regiao)

fig, ax = plt.subplots(figsize=(10, 10))

estados.plot(ax=ax,
             color=estados['cor'],
             edgecolor='white',
             linewidth=0.8)

# Legenda manual
patches = [mpatches.Patch(color=cor, label=reg) for reg, cor in cores_regiao.items()]
ax.legend(handles=patches, loc='lower left', title='Regiões', fontsize=11, title_fontsize=12)

# Adicionar siglas
for _, row in estados.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(row['sigla'],
                xy=(centroid.x, centroid.y),
                ha='center', va='center',
                fontsize=7, color='white', fontweight='bold')

ax.set_title('Brasil — Estados por Região', fontsize=16, fontweight='bold', pad=15)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Dados fictícios de IDH por estado
# Fonte de referência: PNUD Brasil 2021
idh_data = {
    'sigla': ['AC','AL','AM','AP','BA','CE','DF','ES','GO','MA',
              'MG','MS','MT','PA','PB','PE','PI','PR','RJ','RN',
              'RO','RR','RS','SC','SE','SP','TO'],
    'idh':   [0.719,0.683,0.708,0.708,0.714,0.714,0.824,0.740,0.735,0.676,
              0.731,0.729,0.725,0.698,0.700,0.727,0.697,0.756,0.796,0.720,
              0.690,0.707,0.787,0.792,0.720,0.826,0.699]
}

df_idh = pd.DataFrame(idh_data)

# unir o GeoDataFrame com os dados de IDH
estados_idh = estados.merge(df_idh, on='sigla', how='left')

# Calcular área de cada estado

estados_idh_proj = estados_idh.to_crs(epsg=5880)
estados_idh['area_km2'] = estados_idh_proj.geometry.area / 1_000_000

print('10 maiores estados por área (km²):')
estados_idh[['name','sigla','area_km2']].sort_values('area_km2', ascending=False).head(10)

In [ ]:
# Calcular centroides
estados_idh['centroid'] = estados_idh.geometry.centroid

print('Centroides dos estados do Sul:')
sul = estados_idh[estados_idh['regiao'] == 'Sul'][['name','sigla','centroid']]
sul['lon'] = sul['centroid'].x
sul['lat'] = sul['centroid'].y
sul[['name','sigla','lon','lat']]

In [ ]:
# Pegar o estado de São Paulo
sao_paulo = estados_idh[estados_idh['sigla'] == 'SP'].copy()

# Criar um buffer ao redor de SP
buffer_sp = sao_paulo.geometry.buffer(0.5).iloc[0]
vizinhos_sp = estados_idh[estados_idh.geometry.intersects(buffer_sp)]
print('Estados que fazem fronteira de São Paulo:')
print(vizinhos_sp[['name', 'sigla']].to_string(index=False))

In [ ]:
# Visualizar São Paulo e seus vizinhos
fig, ax = plt.subplots(figsize=(9, 7))
estados_idh.plot(ax=ax, color='#e0e0e0', edgecolor='white', linewidth=0.5)

# Vizinhos
vizinhos_sp.plot(ax=ax, color='#90CAF9', edgecolor='white', linewidth=0.8)
sao_paulo.plot(ax=ax, color='#1565C0', edgecolor='white', linewidth=1)

# Buffer
gpd.GeoSeries([buffer_sp]).plot(ax=ax, color='none', edgecolor='red',
                                  linewidth=2, linestyle='--', alpha=0.6)

# Labels
for _, row in vizinhos_sp.iterrows():
    c = row.geometry.centroid
    ax.annotate(row['sigla'], xy=(c.x, c.y), ha='center', va='center',
                fontsize=8, fontweight='bold', color='#333')

ax.set_xlim(-55, -38); ax.set_ylim(-27, -16)
ax.set_title('São Paulo e seus vizinhos\n(buffer de ~55 km)', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

---

Documentação oficial: https://geopandas.org/en/stable/getting_started.html